# 00 — First principles of AI content creation

*Applied Labs · Domain: Prompt Engineering · Advanced · ~30 min*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI2026/castalia/blob/main/labs/07_content_seo_engine/00_first_principles.ipynb)

Content marketing is a **$400 B+** industry where a single quality blog post
costs **$200–500** from a freelancer. AI can compress the research-to-publish
cycle from days to minutes — but only if we understand *why* content ranks,
*how* search intent works, and *what* separates generic filler from content
that converts.

> **Learning goals**
> 1. Map the ranking factors that determine content visibility.
> 2. Quantify why research — not writing — is the real bottleneck.
> 3. Classify search intent types from query features.
> 4. Encode brand voice as retrievable constraints.
> 5. Build a quality rubric that prevents AI content from becoming generic.

## 0 · Setup

In [ ]:
!pip install -q "sentence-transformers>=2.2.0" "numpy>=1.24.0" "textstat>=0.7.0"

In [ ]:
import os
import math
import json
from typing import Any
from collections import Counter

import numpy as np
import textstat

print("Environment ready ✓")

## 1 · What makes content rank?

Google's ranking algorithm evaluates hundreds of signals, but the
**core pillars** that matter most for content creators are:

| Pillar | Signal examples |
|---|---|
| **Topical authority** | Depth of coverage, internal linking, expertise |
| **Search intent match** | Does the page answer what the user actually wants? |
| **E-E-A-T** | Experience, Expertise, Authoritativeness, Trustworthiness |
| **Technical SEO** | Page speed, mobile-friendly, structured data, heading hierarchy |

Let's build an **SEO score checklist** that codifies these factors.

In [ ]:
# SEO ranking factors as a weighted checklist
SEO_RANKING_FACTORS: dict[str, dict[str, Any]] = {
    "topical_authority": {
        "weight": 0.25,
        "checks": [
            "covers_subtopics_comprehensively",
            "includes_related_keywords",
            "internal_links_to_pillar_content",
            "demonstrates_domain_expertise",
        ],
        "description": "Depth and breadth of topic coverage",
    },
    "search_intent_match": {
        "weight": 0.25,
        "checks": [
            "matches_primary_intent_type",
            "answers_query_directly",
            "provides_expected_content_format",
            "addresses_follow_up_questions",
        ],
        "description": "How well the content satisfies the user's goal",
    },
    "eeat_signals": {
        "weight": 0.25,
        "checks": [
            "author_has_credentials",
            "includes_first_hand_experience",
            "cites_authoritative_sources",
            "transparent_about_limitations",
        ],
        "description": "Experience, Expertise, Authoritativeness, Trustworthiness",
    },
    "technical_seo": {
        "weight": 0.25,
        "checks": [
            "keyword_in_title_and_h1",
            "proper_heading_hierarchy",
            "meta_description_optimized",
            "readable_url_structure",
        ],
        "description": "On-page technical optimization",
    },
}


def score_seo_checklist(
    article_checks: dict[str, list[bool]],
    factors: dict[str, dict[str, Any]] = SEO_RANKING_FACTORS,
) -> dict[str, Any]:
    """Score an article against the SEO checklist.

    Parameters
    ----------
    article_checks : dict mapping factor name → list of bool (one per check)
    factors : the ranking factor definitions

    Returns
    -------
    dict with per-factor scores and weighted total.
    """
    results: dict[str, Any] = {}
    total = 0.0
    for factor, meta in factors.items():
        checks = article_checks.get(factor, [False] * len(meta["checks"]))
        pct = sum(checks) / len(checks) if checks else 0.0
        weighted = pct * meta["weight"]
        total += weighted
        results[factor] = {"raw": f"{sum(checks)}/{len(checks)}", "score": round(pct, 2)}
    results["total_weighted_score"] = round(total, 3)
    return results


# Demo: a hypothetical article
demo_checks: dict[str, list[bool]] = {
    "topical_authority":    [True, True, False, True],
    "search_intent_match":  [True, True, True, False],
    "eeat_signals":         [False, True, True, False],
    "technical_seo":        [True, True, True, True],
}

result = score_seo_checklist(demo_checks)
for k, v in result.items():
    print(f"  {k:.<30} {v}")

## 2 · Why research is the bottleneck

Most people assume *writing* is the hard part.  In reality, the time
breakdown for a quality 1,500-word article looks roughly like this:

| Phase | % of time | What it involves |
|---|---|---|
| **Research** | ~60 % | Topic analysis, competitor review, audience questions, data gathering |
| **Writing** | ~30 % | Drafting, structuring, voice application |
| **Editing** | ~10 % | Proofreading, SEO polish, formatting |

This is exactly where LLMs shine: compressing the **research** phase from
hours to minutes while maintaining depth.

In [ ]:
# Topic research framework
def build_research_brief(
    keyword: str,
    audience_questions: list[str],
    competitor_coverage: list[dict[str, str]],
    content_gaps: list[str],
) -> dict[str, Any]:
    """Build a structured research brief for a content topic.

    Parameters
    ----------
    keyword : primary target keyword
    audience_questions : questions the target audience is asking
    competitor_coverage : list of dicts with 'title' and 'angle'
    content_gaps : topics competitors miss

    Returns
    -------
    dict – a structured research brief.
    """
    return {
        "primary_keyword": keyword,
        "audience_questions": audience_questions,
        "competitor_analysis": competitor_coverage,
        "content_gaps": content_gaps,
        "research_depth_score": min(
            len(audience_questions) / 5
            + len(competitor_coverage) / 3
            + len(content_gaps) / 3,
            1.0,
        ),
    }


# Demo
brief = build_research_brief(
    keyword="kubernetes autoscaling",
    audience_questions=[
        "How does HPA differ from VPA?",
        "When should I use cluster autoscaler?",
        "What are the best metrics for custom autoscaling?",
        "How to handle autoscaling with spot instances?",
    ],
    competitor_coverage=[
        {"title": "K8s Autoscaling Guide", "angle": "HPA basics only"},
        {"title": "Scale Your Cluster", "angle": "Cluster autoscaler tutorial"},
    ],
    content_gaps=[
        "No one covers HPA + VPA together",
        "Missing: cost-optimization with autoscaling",
        "No real-world production war stories",
    ],
)
print(json.dumps(brief, indent=2))

## 3 · Search intent types

Every search query carries an **intent** that determines what kind of content
will rank. Getting intent wrong means your content will never surface, no
matter how well-written it is.

| Intent type | User goal | Example query | Content format |
|---|---|---|---|
| **Informational** | Learn something | "what is kubernetes" | Guide, explainer |
| **Navigational** | Find a specific page | "kubernetes docs" | Landing page |
| **Commercial** | Compare options | "best kubernetes tools" | Comparison, review |
| **Transactional** | Take action | "buy kubernetes training" | Product page, CTA |

In [ ]:
# Heuristic intent classifier
INTENT_SIGNALS: dict[str, list[str]] = {
    "informational": [
        "what", "how", "why", "when", "guide", "tutorial",
        "explain", "learn", "definition", "overview",
    ],
    "navigational": [
        "login", "docs", "documentation", "official",
        "homepage", "site", "portal", "dashboard",
    ],
    "commercial": [
        "best", "top", "review", "compare", "vs",
        "alternative", "comparison", "pros and cons",
    ],
    "transactional": [
        "buy", "price", "pricing", "discount", "coupon",
        "free trial", "sign up", "download", "subscribe",
    ],
}


def classify_intent(query: str) -> dict[str, Any]:
    """Classify search intent using keyword heuristics.

    Parameters
    ----------
    query : the user's search query

    Returns
    -------
    dict with intent scores and predicted intent.
    """
    query_lower = query.lower()
    scores: dict[str, int] = {}
    for intent, signals in INTENT_SIGNALS.items():
        scores[intent] = sum(1 for s in signals if s in query_lower)

    total = sum(scores.values()) or 1
    probs = {k: round(v / total, 2) for k, v in scores.items()}
    predicted = max(scores, key=scores.get)  # type: ignore[arg-type]
    return {"query": query, "scores": scores, "probabilities": probs, "predicted_intent": predicted}


# Demo: same keyword space, different intents
test_queries: list[str] = [
    "what is content marketing",
    "HubSpot content marketing login",
    "best content marketing tools 2025",
    "buy content marketing course discount",
]

for q in test_queries:
    r = classify_intent(q)
    print(f"  {q:.<50} → {r['predicted_intent']}")

## 4 · Brand voice as retrieval

Brand voice is not a vague feeling — it's a set of **concrete constraints**:
tone adjectives, vocabulary choices, persona definition, and exemplar
paragraphs. Encoding voice this way makes it *retrievable* and *reproducible*
across hundreds of articles.

In [ ]:
# Brand voice template with few-shot examples
BRAND_VOICES: dict[str, dict[str, Any]] = {
    "technical_b2b": {
        "tone": ["authoritative", "precise", "data-driven"],
        "vocabulary": ["leverage", "optimize", "infrastructure", "scalable"],
        "persona": "Senior engineering lead writing for CTOs and platform teams",
        "example_paragraph": (
            "Horizontal pod autoscaling in Kubernetes leverages real-time "
            "metric streams to dynamically adjust replica counts. Our "
            "benchmarks across 14 production clusters show a 37% reduction "
            "in compute waste when combining HPA with custom Prometheus "
            "metrics — without sacrificing p99 latency targets."
        ),
    },
    "friendly_consumer": {
        "tone": ["conversational", "encouraging", "clear"],
        "vocabulary": ["easy", "simple", "step-by-step", "you'll love"],
        "persona": "Helpful friend who makes complex topics approachable",
        "example_paragraph": (
            "Ever feel like your cloud bill is out of control? You're not "
            "alone! The good news is, Kubernetes has a built-in trick called "
            "autoscaling that can save you serious money. Think of it like a "
            "smart thermostat for your servers — it dials resources up when "
            "things get busy and back down when they don't."
        ),
    },
    "thought_leadership": {
        "tone": ["visionary", "provocative", "insightful"],
        "vocabulary": ["paradigm", "inflection point", "rethink", "first principles"],
        "persona": "Industry analyst challenging conventional wisdom",
        "example_paragraph": (
            "The autoscaling conversation is stuck in 2019. While the "
            "industry obsesses over replica counts, the real inflection "
            "point is intent-based scaling — where infrastructure anticipates "
            "demand from business signals, not trailing CPU metrics. "
            "Organizations that rethink scaling from first principles will "
            "define the next era of cloud-native efficiency."
        ),
    },
}


def build_voice_prompt(voice_name: str, topic: str) -> str:
    """Build a writing prompt that encodes brand voice constraints.

    Parameters
    ----------
    voice_name : key into BRAND_VOICES
    topic : the article topic

    Returns
    -------
    str – a system-level prompt fragment.
    """
    voice = BRAND_VOICES[voice_name]
    return (
        f"Write about '{topic}' in a {', '.join(voice['tone'])} tone.\n"
        f"Persona: {voice['persona']}\n"
        f"Preferred vocabulary: {', '.join(voice['vocabulary'])}\n"
        f"Style example:\n\"\"{voice['example_paragraph']}\"\""
    )


# Show 3 voices on the same topic
topic = "Kubernetes autoscaling"
for name in BRAND_VOICES:
    prompt = build_voice_prompt(name, topic)
    print(f"─── {name} ───")
    print(prompt[:200], "…\n")

## 5 · Quality eval as differentiator

Without rigorous evaluation, AI-generated content quickly converges to
**generic filler**: grammatically correct but shallow, unoriginal, and
interchangeable. The quality rubric is what turns an LLM from a text
generator into a content *engine*.

In [ ]:
# Content quality rubric
QUALITY_RUBRIC: dict[str, dict[str, Any]] = {
    "depth": {
        "weight": 0.25,
        "levels": {
            5: "Expert-level detail with data, examples, edge cases",
            4: "Solid coverage with some specifics",
            3: "Surface-level treatment of the topic",
            2: "Thin content missing key subtopics",
            1: "Stub or placeholder content",
        },
    },
    "originality": {
        "weight": 0.20,
        "levels": {
            5: "Unique angle, original research or insights",
            4: "Fresh framing of existing information",
            3: "Standard treatment, nothing new",
            2: "Mostly rehashed from obvious sources",
            1: "Clearly templated / generic",
        },
    },
    "readability": {
        "weight": 0.20,
        "levels": {
            5: "Engaging, clear, well-structured",
            4: "Easy to follow with good flow",
            3: "Readable but dry or uneven",
            2: "Dense, jargon-heavy, hard to parse",
            1: "Confusing or poorly organized",
        },
    },
    "seo_compliance": {
        "weight": 0.20,
        "levels": {
            5: "All SEO best practices met",
            4: "Most SEO requirements satisfied",
            3: "Basic SEO present, some gaps",
            2: "Minimal SEO effort",
            1: "No SEO optimization",
        },
    },
    "voice_consistency": {
        "weight": 0.15,
        "levels": {
            5: "Perfectly on-brand throughout",
            4: "Mostly consistent with minor drift",
            3: "Inconsistent in places",
            2: "Frequently off-brand",
            1: "No discernible brand voice",
        },
    },
}


def score_article(
    scores: dict[str, int],
    rubric: dict[str, dict[str, Any]] = QUALITY_RUBRIC,
) -> dict[str, Any]:
    """Compute weighted quality score for an article.

    Parameters
    ----------
    scores : dict mapping dimension → integer score (1-5)
    rubric : the quality rubric definitions

    Returns
    -------
    dict with per-dimension weighted scores and total.
    """
    details: dict[str, Any] = {}
    total = 0.0
    for dim, meta in rubric.items():
        raw = scores.get(dim, 3)
        weighted = (raw / 5.0) * meta["weight"]
        total += weighted
        details[dim] = {"raw": raw, "weighted": round(weighted, 3)}
    details["total_score"] = round(total, 3)
    details["grade"] = (
        "A" if total >= 0.85 else
        "B" if total >= 0.70 else
        "C" if total >= 0.55 else
        "D" if total >= 0.40 else "F"
    )
    return details


# Demo: two hypothetical articles
articles = {
    "Well-researched piece": {"depth": 5, "originality": 4, "readability": 4, "seo_compliance": 5, "voice_consistency": 4},
    "Generic AI filler":     {"depth": 2, "originality": 1, "readability": 3, "seo_compliance": 2, "voice_consistency": 2},
}

for title, scores in articles.items():
    result = score_article(scores)
    print(f"  {title}: {result['total_score']:.3f} (Grade {result['grade']})")

## 6 · Market analysis

| Metric | Value |
|---|---|
| Content marketing TAM | **$400 B+** globally |
| Average freelancer blog post | **$200–500** |
| Jasper AI ARR | **$80 M+** |
| Writer AI ARR | **$100 M+** |
| Posts per month (mid-size co.) | **20–50** |
| Monthly spend at $300/post | **$6,000–15,000** |

The bottleneck is **research + consistency**, not raw word generation.
Companies that can produce research-informed, brand-consistent content
at scale capture an outsized share of organic traffic — and the
economics improve by **5–10×** with AI-assisted workflows.

In [ ]:
# Market economics calculator
def content_roi(
    posts_per_month: int = 30,
    freelancer_cost: float = 350.0,
    ai_cost_per_post: float = 2.50,
    avg_monthly_traffic_per_post: int = 200,
    value_per_visit: float = 0.50,
) -> dict[str, Any]:
    """Calculate ROI of AI content engine vs. freelancers.

    Parameters
    ----------
    posts_per_month : target publication volume
    freelancer_cost : average cost per freelance article
    ai_cost_per_post : LLM API cost per article
    avg_monthly_traffic_per_post : organic visits per post per month
    value_per_visit : revenue per organic visit

    Returns
    -------
    dict with cost comparison and ROI metrics.
    """
    freelancer_monthly = posts_per_month * freelancer_cost
    ai_monthly = posts_per_month * ai_cost_per_post
    savings = freelancer_monthly - ai_monthly
    monthly_traffic = posts_per_month * avg_monthly_traffic_per_post
    monthly_revenue = monthly_traffic * value_per_visit
    return {
        "freelancer_monthly_cost": f"${freelancer_monthly:,.0f}",
        "ai_monthly_cost": f"${ai_monthly:,.0f}",
        "monthly_savings": f"${savings:,.0f}",
        "cost_reduction": f"{(savings / freelancer_monthly) * 100:.1f}%",
        "monthly_traffic_generated": f"{monthly_traffic:,}",
        "monthly_revenue_from_content": f"${monthly_revenue:,.0f}",
        "annual_savings": f"${savings * 12:,.0f}",
    }


roi = content_roi()
print("Content Engine ROI Analysis")
print("=" * 40)
for k, v in roi.items():
    print(f"  {k:.<35} {v}")

## Exercises

1. **Custom rubric dimension** — Add a "factual accuracy" dimension to
   `QUALITY_RUBRIC` with 5 levels and a weight of 0.15 (re-normalise the
   others). Score both demo articles.
2. **Intent classifier v2** — Extend `classify_intent` to handle
   *multi-intent* queries (e.g., "best kubernetes course buy") that blend
   commercial + transactional signals. Return the top-2 intents with
   confidence.
3. **Voice distance metric** — Using `sentence-transformers`, compute
   cosine similarity between each brand voice example paragraph and a
   new paragraph you write. Which voice is closest?

## Takeaways

* Content ranking depends on **topical authority**, **intent match**,
  **E-E-A-T**, and **technical SEO** — not just word count.
* **Research** consumes ~60 % of content creation time — the biggest
  leverage point for AI.
* Search intent classification is a prerequisite for content that ranks.
* Brand voice is a set of **retrievable constraints**, not a feeling.
* Without a **quality rubric**, AI content degrades to generic filler.

## What's next

In **01 — Architecture** we'll design the full content engine pipeline:
topic research → brief generation → writing → SEO optimization →
quality evaluation → revision loop.

---

## References

1. Google Search Central — "How Search Works" (2024)
   https://developers.google.com/search/docs/fundamentals/how-search-works
2. Backlinko — "Google Ranking Factors: The Complete List" (2024)
   https://backlinko.com/google-ranking-factors
3. HubSpot — "The State of Content Marketing in 2024"
   https://www.hubspot.com/state-of-marketing
4. Flesch, R. — *The Art of Readable Writing* (1949)